In [ ]:
!pip install pysam
!pip install statsmodels

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import statistics
import matplotlib.pyplot as plt
from scipy.stats import binomtest
import statsmodels.formula.api as smf
import statsmodels.api as sm
from inference import COSMIC, EM
import pysam
import pyspark
import dxpy
import dxdata

In [ ]:
macs = {}

for chrom in tqdm(range(1, 23)):

    path = f'/mnt/project/DNM/trios/diffs/chr{chrom}/'

    with open(f'{path}trios_chr{chrom}.txt') as f:

        for row in f:

            row = row.strip()
            row = row.split(' ')

            diff_id = row[0]
            diff_mac = int(row[1])
            diff_obs = int(row[2])

            macs[diff_id] = (diff_mac, diff_obs)

In [ ]:
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa'
!dx download 'Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai'
fasta = pysam.FastaFile('GRCh38_full_analysis_set_plus_decoy_hla.fa')

## QC

In [ ]:
dp_lower_thre = 20
dp_upper_thre = 100

gq_thre = 30
p_thre = 0.05

In [ ]:
inds_diffs_qc = {}

path = f'/mnt/project/DNM/trios/QC/'

for b in tqdm(range(11)):
    
    with open(f'{path}trios_qc_b{b}.txt') as f:

        for row in f:
            
            row = row.strip()
            row = row.split(' ')

            ind_idx = str(row[0])
            if ind_idx not in inds_diffs_qc:
                inds_diffs_qc[ind_idx] = []

            chrom = int(row[1].split(':')[0])
            pos = int(row[1].split(':')[1])
            ref = row[1].split(':')[2]
            alt = row[1].split(':')[3]
                
            mac, obs = macs[row[1]]

            ref_ad = int(row[2])
            alt_ad = int(row[3])
            dp = int(row[4])
            gq = int(row[5])

            father_ref_ad = int(row[8])
            father_alt_ad = int(row[9])
            father_dp = int(row[10])
            father_gq = int(row[11])
            
            mother_ref_ad = int(row[14])
            mother_alt_ad = int(row[15])
            mother_dp = int(row[16])
            mother_gq = int(row[17])

            if (dp >= dp_lower_thre and father_dp >= dp_lower_thre and mother_dp >= dp_lower_thre and
                gq >= gq_thre and father_gq >= gq_thre and mother_gq >= gq_thre and
                dp <= dp_upper_thre and father_alt_ad == 0 and mother_alt_ad == 0):

                p = binomtest(alt_ad, dp, 0.5).pvalue

                if p > p_thre:
                    inds_diffs_qc[ind_idx].append((chrom, pos, ref, alt,
                                                   mac, obs, alt_ad, dp, gq,
                                                   father_alt_ad, father_dp, father_gq,
                                                   mother_alt_ad, mother_dp, mother_gq))

In [ ]:
with open('trios_diffs_info_v0.txt', 'w') as f:
    f.write('IID\tCHROM\tPOS\tref\talt\tMAC\tOBS\n')
    for ind in inds_diffs_qc:
        for diff in inds_diffs_qc[ind]:
            f.write(f'{ind}\t{diff[0]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t{diff[4]}\t{diff[5]}\n')

In [ ]:
!dx upload trios_diffs_info_v0.txt --path='DNM/trios/out/'

In [ ]:
# inds_diffs_qc_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info_v0.txt', sep = '\t')

# inds_diffs_qc = {}

# for _, row in inds_diffs_qc_df.iterrows():
#     if row['IID'] not in inds_diffs_qc:
#         inds_diffs_qc[row['IID']] = []
#     inds_diffs_qc[row['IID']].append((row['CHROM'], row['POS'], row['ref'], 
#                                       row['alt'], row['MAC'], row['OBS']))

In [ ]:
cnts = [len(inds_diffs_qc[ind_idx]) for ind_idx in inds_diffs_qc]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per child')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per child')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Exclude clusters

In [ ]:
def exclude_clusters(ind_diffs, window = 100000, thre = 3):
    
    ind_diffs = sorted(ind_diffs, key = lambda x: (x[0], x[1]))
    
    ind_diffs_chrom = {chrom: [] for chrom in range(1, 23)}
    for diff in ind_diffs:
        ind_diffs_chrom[diff[0]].append(diff)
        
    ind_diffs_exc = []
    
    for chrom in ind_diffs_chrom:
        
        pointer = 0
    
        for i in range(len(ind_diffs_chrom[chrom])):

            while (ind_diffs_chrom[chrom][i][0] == ind_diffs_chrom[chrom][pointer][0] and 
                   (ind_diffs_chrom[chrom][i][1]-ind_diffs_chrom[chrom][pointer][1]) > window):
                pointer += 1

            if (i-pointer+1) >= thre:
                for j in range(pointer, i+1):
                    ind_diffs_exc.append(ind_diffs_chrom[chrom][j])

    return [diff for diff in ind_diffs if diff not in ind_diffs_exc]

In [ ]:
inds_diffs_cluster = {ind_idx: exclude_clusters(inds_diffs_qc[ind_idx], window = 0) for ind_idx in inds_diffs_qc}

In [ ]:
cnts = [len(inds_diffs_cluster[ind_idx]) for ind_idx in inds_diffs_cluster]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per child')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per child')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Exclude outliers

In [ ]:
inds_diffs_out = {ind_idx: inds_diffs_cluster[ind_idx] for ind_idx in inds_diffs_cluster 
                  if len(inds_diffs_cluster[ind_idx]) < 900}

In [ ]:
cnts = [len(inds_diffs_out[ind_idx]) for ind_idx in inds_diffs_out]
                
print(min(cnts), max(cnts), round(statistics.mean(cnts), 2), statistics.median(cnts), sum(cnts), len(cnts))

In [ ]:
plt.boxplot(cnts)
plt.ylabel('Number of SNP differences per child')
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of SNP differences per child')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
plt.hist(cnts, bins = 20)
plt.xlabel('Number of de novo SNPs per child')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Effect of AF filter

In [ ]:
def get_types(inds_diffs):

    types = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        types[f'{prev}[{ref}>{alt}]{next}'] = 0

    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

    for ind in inds_diffs:
        for diff in inds_diffs[ind]:

            chrom = diff[0]
            pos = diff[1]
            ref = diff[2]
            alt = diff[3]

            cont = fasta.fetch(f'chr{chrom}', pos-2, pos+1)
            tp = f'{cont[0]}[{ref}>{alt}]{cont[2]}'

            if ref == 'A' or ref == 'G':
                ref = comp[ref]
                alt = comp[alt]
                cont = f'{comp[cont[0]]}{comp[cont[1]]}{comp[cont[2]]}'
                cont = cont[::-1]
                tp = (comp[tp[6]]+tp[1]+comp[tp[2]]+tp[3]+ 
                       comp[tp[4]]+tp[5]+comp[tp[0]])

            types[tp] += 1

    return types

In [ ]:
def get_types_part(types):
    
    types_part = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                types_part[f'{ref}>{alt}'] = 0
                if ref == 'C' and alt == 'T':
                    types_part['CpG>TpG'] = 0
                        
    for tp in types:
        tp_part = tp[2:5]
        if tp_part[0] == 'C' and tp_part[2] == 'T' and tp[-1] == 'G':
            tp_part = 'CpG>TpG'
        types_part[tp_part] += types[tp]
            
    return types_part

In [ ]:
def exclude_common(ind_diffs, maf = 0.001):
    
    return [diff for diff in ind_diffs if float(diff[4])/float(diff[5]) <= maf]

In [ ]:
inds_diffs_maf = {ind_idx: exclude_common(inds_diffs_out[ind_idx]) for ind_idx in inds_diffs_out}

In [ ]:
inds_types_out = get_types(inds_diffs_out)
inds_types_part_out = get_types_part(inds_types_out)
inds_types_part_out

In [ ]:
inds_types_maf = get_types(inds_diffs_maf)
inds_types_part_maf = get_types_part(inds_types_maf)
inds_types_part_maf

In [ ]:
for tp in inds_types_part_maf:
    print(tp, round((1-inds_types_part_maf[tp]/inds_types_part_out[tp])*100, 2))

# Effect of parental age

In [ ]:
# Clinical data
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

dispensed_database_name = dxpy.find_one_data_object(classname = 'database', name = 'app*', folder = '/', name_mode = 'glob', describe = True)['describe']['name']
dispensed_dataset_id = dxpy.find_one_data_object(typename = 'Dataset', name = 'app*.dataset', folder = '/', name_mode = 'glob')['id']

dataset = dxdata.load_dataset(id = dispensed_dataset_id)
participant = dataset['participant']

field_names = ['eid', 'p34']

df = participant.retrieve_fields(names = field_names, engine = dxdata.connect())

inds_sql_df = df.toPandas()
inds_sql_df['eid'] = inds_sql_df['eid'].astype(int)
inds_sql_df = inds_sql_df.dropna(subset = ['p34'])
inds_sql_df.head()

In [ ]:
ind_yob = inds_sql_df.set_index('eid')['p34'].to_dict()

In [ ]:
data = []

with open('/mnt/project/DNM/trios_pairs.txt') as f:
    
    for i, row in enumerate(f):
        
        row = row.strip()
        row = row.split(' ')
    
        child = int(row[0])
        father = int(row[1])
        mother = int(row[2])
        
        b = int(i/100)
        p = i%100
            
        child_idx = f'{child}_b{b}_p{p}'
        father_idx = f'{father}_b{b}_p{p}'
        mother_idx = f'{mother}_b{b}_p{p}'
        
        if child_idx not in inds_diffs_out:
            continue
        
        data.append((ind_yob[child]-ind_yob[father],
                     ind_yob[child]-ind_yob[mother],
                     len(inds_diffs_out[child_idx])))

In [ ]:
data_df = pd.DataFrame(data, columns = ['father_age', 'mother_age', 'dnm_cnt'])

model = smf.glm(formula = 'dnm_cnt ~ father_age + mother_age', data = data_df, 
                family = sm.families.Poisson(sm.families.links.Identity())).fit()
model.summary()

In [ ]:
mother_age_mean = data_df['mother_age'].mean()
father_age_grid = np.linspace(data_df['father_age'].min(), data_df['father_age'].max(), 100)

df = pd.DataFrame({
    'father_age': father_age_grid,
    'mother_age': mother_age_mean
})

ci = model.get_prediction(df).summary_frame(alpha = 0.05)

plt.fill_between(father_age_grid, ci['mean_ci_lower'], ci['mean_ci_upper'],
                 color = 'lightgrey', alpha = 0.2, label = '95% CI')

plt.scatter(data_df['father_age'], data_df['dnm_cnt'], alpha = 0.7)
plt.plot(father_age_grid, ci['mean'])

plt.xlabel('Paternal age at birth')
plt.ylabel("Child's de novo SNP count")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
father_age_mean = data_df['father_age'].mean()
mother_age_grid = np.linspace(data_df['mother_age'].min(), data_df['mother_age'].max(), 100)

df = pd.DataFrame({
    'mother_age': mother_age_grid,
    'father_age': father_age_mean
})

ci = model.get_prediction(df).summary_frame(alpha = 0.05)

plt.fill_between(mother_age_grid, ci['mean_ci_lower'], ci['mean_ci_upper'],
                 color = 'lightgrey', alpha = 0.2, label = '95% CI')

plt.scatter(data_df['mother_age'], data_df['dnm_cnt'], alpha = 0.7)
plt.plot(mother_age_grid, ci['mean'])

plt.xlabel('Maternal age at birth')
plt.ylabel("Child's de novo SNP count")
plt.grid(True)
plt.legend()
plt.show()

# Gene conversion events

In [ ]:
dmc1 = {}

dmc1_df = pd.read_csv('/mnt/project/DNM/dmc1_hotspots.csv')

for _, row in dmc1_df.iterrows():
    
    if (row['Chromosome'] == 'chrX' or 
        row['Chromosome'] == 'chrY'):
        continue
    
    chrom = int(row['Chromosome'][3:])
    start = int(row['Start_Pos'])
    end = int(row['End_Pos'])
    
    if chrom not in dmc1:
        dmc1[chrom] = []
    dmc1[chrom].append((start, end))

In [ ]:
cnt = 0
total = 0

for ind in inds_diffs_out:
    
    for diff in inds_diffs_out[ind]:
        
        in_dmc1 = False
        
        for start, end in dmc1[diff[0]]:
            if diff[1] >= start and diff[1] <= end:
                in_dmc1 = True
                
        if in_dmc1 == True:
            cnt += 1
        total += 1
        
print(cnt, total, round(cnt/total, 4))

# Mutational spectrum and signatures

In [ ]:
def get_types(diffs):

    muts = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        muts[f'{prev}[{ref}>{alt}]{next}'] = 0

    comp = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

    for diff in diffs:

        chrom = diff[0]
        pos = diff[1]
        ref = diff[2]
        alt = diff[3]

        cont = fasta.fetch(f'chr{chrom}', pos-2, pos+1)
        mut = f'{cont[0]}[{ref}>{alt}]{cont[2]}'

        if ref == 'A' or ref == 'G':
            ref = comp[ref]
            alt = comp[alt]
            cont = f'{comp[cont[0]]}{comp[cont[1]]}{comp[cont[2]]}'
            cont = cont[::-1]
            mut = (comp[mut[6]]+mut[1]+comp[mut[2]]+mut[3]+ 
                   comp[mut[4]]+mut[5]+comp[mut[0]])

        muts[mut] += 1

    return muts

In [ ]:
def merge_types(inds_types):
    
    muts = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        muts[f'{prev}[{ref}>{alt}]{next}'] = 0
                        
    for ind in inds_types:
        for mut in inds_types[ind]:
            muts[mut] += inds_types[ind][mut]
            
    muts = {mut: muts[mut]/sum(muts.values()) for mut in muts}
            
    return muts

In [ ]:
def list_types(inds_types):
    
    muts = []
    
    for ind in inds_types:
        for mut in inds_types[ind]:
            for i in range(inds_types[ind][mut]):
                muts.append(mut)
            
    muts = pd.DataFrame({'Type': muts})
    
    return muts

In [ ]:
def get_types_part(inds_types):
    
    types_part = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                types_part[f'{ref}>{alt}'] = 0
                if ref == 'C' and alt == 'T':
                    types_part['CpG>TpG'] = 0
                        
    for tp in inds_types:
        tp_part = tp[2:5]
        if tp_part[0] == 'C' and tp_part[2] == 'T' and tp[-1] == 'G':
            tp_part = 'CpG>TpG'
        types_part[tp_part] += inds_types[tp]
            
    return types_part

In [ ]:
def plot_spectrum(spectrum, ylim = None):

    colors = {
        'C>A': '#1f77b4',
        'C>G': '#000000',
        'C>T': '#e41a1c',
        'T>A': '#999999',
        'T>C': '#4daf4a',
        'T>G': '#ff7f00'
    }

    types = list(spectrum.keys())
    fracts = list(spectrum.values())
    groups = [colors[group[2:5]] for group in spectrum]
    
    fig, ax = plt.subplots(figsize = (12, 3))
    bars = ax.bar(types, fracts, color = groups, width = 0.8)
    ax.set_xticks(range(len(types)))
    ax.set_xticklabels(types, rotation = 90, fontsize = 5)
    ax.margins(x = 0)
    
    if ylim:
        ax.set_ylim(0, ylim)

    ax.set_ylabel('Distribution')
    ax.set_xlabel('Mutation type')
    plt.show()
    plt.clf()

In [ ]:
def plot_spectrum_part(spectrum):

    types = list(spectrum.keys())
    fracts = [cnt/sum(spectrum.values()) 
              for cnt in spectrum.values()]
    
    x = list(range(len(spectrum)))
    plt.bar(x, fracts)
    plt.ylabel('Distribution')
    plt.xlabel('Mutation type')
    plt.xticks(x, types)
    plt.show()
    plt.clf()

In [ ]:
def plot_signatures(spectrum, ylim = None):
    
    em = EM(spectrum, COSMIC(), beta = 0)
    em.infer()
    em.plot(first = 5, ylim = ylim)
    return em.fracts()

In [ ]:
inds_types = {ind: get_types(inds_diffs_out[ind]) for ind in inds_diffs_out}

In [ ]:
spectrum = merge_types(inds_types)
plot_spectrum(spectrum)

In [ ]:
spectrum = get_types_part(merge_types(inds_types))
plot_spectrum_part(spectrum)

In [ ]:
spectrum = list_types(inds_types)
print(plot_signatures(spectrum))

# Record differences

In [ ]:
with open('trios_diffs_info.txt', 'w') as f:
    f.write('IID\tCHROM\tPOS\tref\talt\tMAC\tOBS\n')
    for ind in inds_diffs_out:
        for diff in inds_diffs_out[ind]:
            f.write(f'{ind}\t{diff[0]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t{diff[4]}\t{diff[5]}\n')

In [ ]:
spectrum = merge_types(inds_types)
with open('trios_diffs_types.txt', 'w') as f:
    f.write('Type\tFraction\n')
    for type in spectrum:
        f.write(f'{type}\t{spectrum[type]}\n')

In [ ]:
with open('trios_diffs_sigs.txt', 'w') as f:
    f.write('Project\tSample\tID\tGenome\tmut_type\tchrom\tpos_start\tpos_end\tref\talt\tType\n')
    for idx, ind in enumerate(inds_diffs_out):
        for diff in inds_diffs_out[ind]:
            f.write(f'.\t{idx}\t.\tGRCh38\tSNP\t{diff[0]}\t{diff[1]}\t{diff[1]}\t{diff[2]}\t{diff[3]}\t.\n')

In [ ]:
with open('trios_diffs_vep.vcf', 'w') as f:
    for ind in inds_diffs_out:
        for diff in inds_diffs_out[ind]:
            f.write(f'{diff[0]} {diff[1]} {diff[2]} {diff[3]}\n')

In [ ]:
!dx upload 'trios_diffs_info.txt' --path='DNM/trios/out/'
!dx upload 'trios_diffs_types.txt' --path='DNM/trios/out/'
!dx upload 'trios_diffs_sigs.txt' --path='DNM/trios/out/'
!dx upload 'trios_diffs_vep.vcf' --path='DNM/trios/out/'